# LO benchmark (linear optimization)

Runs prompts from `data/lp/benchmark.csv`: **9 vignettes**.

Each prompt is a small linear-optimization word problem where some constraints are left implicit (integrality and/or non-negativity). The model must reply with JSON:

```json
{"solution": {...}, "cost": <number>}
```

**Scoring:** `score=true` when the parsed `cost` is within **1%** of the keyed objective. The merged CSV also flags **`naive_lp_confusion`** when `cost` matches the stated-constraints-only optimum.

Tune **`DEBUG_MAX_PROMPTS`** and **`MAX_OUTPUT_TOKENS`** in the setup cell. Default output cap is **1024** tokens.

**Kaggle setup:** Add-ons → Secrets → `GITHUB_TOKEN` (GitHub PAT with `repo` scope, toggled ON). Settings → Internet ON. **Run the setup cell below first.**

**Publishing (Build task):** In the setup cell set `SKIP_DRY_RUN_FOR_BUILD = True` to save compute, then click **Build task**. The publish cell calls `lo_normative_accuracy_5.run()` to create the required `.run.json`, then `%choose`.

**Outputs:** The dry-run cell writes `lp_merged_results.csv` and `lp_rate_score_pivot.csv` to `/kaggle/working/`.

In [ ]:
import csv
import io
import shutil
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

GITHUB_OWNER = "FrancisGanong-N"
GITHUB_REPO = "sceptical_llms"
GITHUB_BRANCH = "master"
KAGGLE_REPO_DIR = Path("/kaggle/working") / GITHUB_REPO
EXPECTED_BENCHMARK_PROMPTS = 36
FORCE_REPO_REFRESH = True
DEBUG_MAX_PROMPTS = None
MAX_OUTPUT_TOKENS = 1024
SKIP_DRY_RUN_FOR_BUILD = False  # True before Build task to skip preview cells
CANDIDATE_LLMS = [
    "anthropic/claude-opus-5@default",
    # "openai/gpt-5.6-sol",
]


def benchmark_prompt_count(root: Path) -> int:
    benchmark_csv = root / "data" / "lp" / "benchmark.csv"
    if not benchmark_csv.is_file():
        return 0
    with benchmark_csv.open(newline="", encoding="utf-8") as handle:
        return sum(1 for _ in csv.DictReader(handle))


def has_text_only_modalities(root: Path) -> bool:
    simple_py = root / "benchmarks" / "simple_rate_tasks.py"
    if not simple_py.is_file():
        return False
    return '"modalities"' in simple_py.read_text(encoding="utf-8")


def has_null_message_patch(root: Path) -> bool:
    patch_py = root / "benchmarks" / "kbench_openai_patch.py"
    tasks_py = root / "benchmarks" / "lp_rate_tasks.py"
    if not patch_py.is_file() or not tasks_py.is_file():
        return False
    return "apply_null_message_patch" in tasks_py.read_text(encoding="utf-8")


def has_fresh_repo(root: Path) -> bool:
    lp_rate_py = root / "benchmarks" / "lp_rate.py"
    tasks_py = root / "benchmarks" / "lp_rate_tasks.py"
    if not tasks_py.is_file():
        return False
    tasks_source = tasks_py.read_text(encoding="utf-8")
    return (
        lp_rate_py.is_file()
        and benchmark_prompt_count(root) >= EXPECTED_BENCHMARK_PROMPTS
        and "parse_lp_json" in lp_rate_py.read_text(encoding="utf-8")
        and "lo_normative_accuracy_5" in tasks_source
        and "_prompt_llm" in tasks_source
        and "response = llm.prompt(prompt)" not in tasks_source
        and has_text_only_modalities(root)
        and has_null_message_patch(root)
    )


def repo_has_quota_safe_prompting(root: Path) -> bool:
    tasks_py = root / "benchmarks" / "lp_rate_tasks.py"
    if not tasks_py.is_file():
        return False
    source = tasks_py.read_text(encoding="utf-8")
    return (
        "_prompt_llm" in source
        and "response = llm.prompt(prompt)" not in source
        and has_text_only_modalities(root)
        and has_null_message_patch(root)
    )


def download_repo_from_github() -> Path:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = (
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
        f"/zipball/{GITHUB_BRANCH}"
    )
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "kaggle-sceptical-llms-lo-benchmark",
        },
    )

    staging = Path("/kaggle/working") / "_repo_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(staging)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub download failed ({exc.code}) for "
            f"github.com/{GITHUB_OWNER}/{GITHUB_REPO}@{GITHUB_BRANCH}: {body[:300]}"
        ) from exc

    extracted = next(p for p in staging.iterdir() if p.is_dir())
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    shutil.copytree(extracted, KAGGLE_REPO_DIR)
    shutil.rmtree(staging)
    return KAGGLE_REPO_DIR


def bootstrap_repo() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if has_fresh_repo(candidate):
            return candidate

    on_kaggle = Path("/kaggle/working").is_dir()
    if on_kaggle:
        if not FORCE_REPO_REFRESH and has_fresh_repo(KAGGLE_REPO_DIR):
            return KAGGLE_REPO_DIR
        root = download_repo_from_github()
        if not repo_has_quota_safe_prompting(root):
            raise RuntimeError(
                "GitHub download succeeded but benchmarks/lp_rate_tasks.py is "
                "missing quota-safe prompting. Push latest master, then re-run."
            )
        return root

    raise RuntimeError(
        "Could not find a fresh sceptical-llms repo (need "
        f"{EXPECTED_BENCHMARK_PROMPTS} rows in data/lp/benchmark.csv). "
        "Run from the repo (or its parent), or on Kaggle set GITHUB_TOKEN."
    )


ROOT = bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for module_name in list(sys.modules):
    if module_name == "benchmarks" or module_name.startswith("benchmarks."):
        del sys.modules[module_name]

print("Repo root:", ROOT)
print("Benchmark rows:", benchmark_prompt_count(ROOT))
print("Repo has LO benchmark support:", has_fresh_repo(ROOT))
print("Repo has quota-safe prompting:", repo_has_quota_safe_prompting(ROOT))
print("Text-only modalities:", has_text_only_modalities(ROOT))
print("Null-message patch:", has_null_message_patch(ROOT))
print("DEBUG_MAX_PROMPTS:", DEBUG_MAX_PROMPTS)
print("MAX_OUTPUT_TOKENS:", MAX_OUTPUT_TOKENS)
if not repo_has_quota_safe_prompting(ROOT):
    raise RuntimeError(
        "Stale benchmarks/lp_rate_tasks.py. Set FORCE_REPO_REFRESH = True "
        "and re-run this setup cell."
    )

In [ ]:
import csv
from pathlib import Path

if SKIP_DRY_RUN_FOR_BUILD:
    print("Skipping dry-run preview (SKIP_DRY_RUN_FOR_BUILD=True)")
else:
    import kaggle_benchmarks as kbench
    from benchmarks.lp_rate import print_score_pivots
    from benchmarks.lp_rate_tasks import evaluate_lp_rate_benchmark

    tasks_source = (Path(ROOT) / "benchmarks" / "lp_rate_tasks.py").read_text(encoding="utf-8")
    if "response = llm.prompt(prompt)" in tasks_source:
        raise RuntimeError(
            "Stale lp_rate_tasks.py on disk. Re-run the setup cell "
            "(FORCE_REPO_REFRESH = True) before this cell."
        )

    def _brief_error(exc: BaseException) -> str:
        message = str(exc).strip().splitlines()[0] if str(exc).strip() else type(exc).__name__
        if len(message) > 200:
            message = message[:197] + "..."
        return f"{type(exc).__name__}: {message}"

    available = set(kbench.llms.keys())
    llm_errors: list[str] = []
    selected_llm_name = None
    runs = score = merged_path = pivot_path = pivot = naive_rate = variant_scores = None

    for llm_name in CANDIDATE_LLMS:
        if llm_name not in available:
            summary = f"{llm_name}: KeyError (not in kbench.llms)"
            llm_errors.append(summary)
            print(summary)
            continue
        print(f"Trying {llm_name}...")
        try:
            (
                runs,
                score,
                merged_path,
                pivot_path,
                pivot,
                naive_rate,
                variant_scores,
            ) = evaluate_lp_rate_benchmark(
                kbench.llms[llm_name],
                max_prompts=DEBUG_MAX_PROMPTS,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                n_jobs=1,
            )
            selected_llm_name = llm_name
            print(f"Using {llm_name}")
            break
        except Exception as exc:
            summary = f"{llm_name}: {_brief_error(exc)}"
            llm_errors.append(summary)
            print(summary)

    if selected_llm_name is None:
        raise RuntimeError(
            "All candidate LLMs failed:\n" + "\n".join(llm_errors)
        )

    print("Overall keyed accuracy:", f"{score.accuracy:.1%}")
    print("JSON solve accuracy:", f"{variant_scores['json']:.1%}")
    print("Needs tacit accuracy:", f"{variant_scores['needs_tacit_constraint']:.1%}")
    print("Detects violation accuracy:", f"{variant_scores['detects_tacit_violation']:.1%}")
    print("Naive LP confusion rate:", f"{naive_rate:.1%}")
    print("Parse rate:", f"{score.parse_rate:.1%}")
    print("Merged results:", merged_path)
    print("Score pivot CSV:", pivot_path)

    import pandas as pd

    merged_df = pd.read_csv(merged_path)
    expected_n = DEBUG_MAX_PROMPTS or EXPECTED_BENCHMARK_PROMPTS
    empty_responses = int(
        merged_df["response"].astype(str).str.strip().eq("").sum()
    ) if "response" in merged_df.columns else 0
    blank_scores = int(
        merged_df["score"].isna().sum()
        + (
            merged_df["score"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["", "nan", "none"])
            & merged_df["score"].notna()
        ).sum()
    )
    pivot_nan = int(pivot.isna().sum().sum()) if pivot is not None else 0
    print(
        f"Coverage: {len(merged_df) - empty_responses} answered / "
        f"{len(merged_df)} merged rows / {expected_n} expected prompts"
    )
    print(f"Empty responses (missing scores): {empty_responses}")
    print(f"Blank/NaN score fields: {blank_scores}")
    print(f"NaN cells in score pivot: {pivot_nan} / {pivot.size if pivot is not None else 0}")

    with merged_path.open(encoding="utf-8") as handle:
        print_score_pivots(list(csv.DictReader(handle)))

    try:
        from IPython.display import display

        display(pivot)
    except ImportError:
        pass


## Download results for local analysis

Run the cell below to inspect paths and preview the merged CSV. Copy results to `data/kaggle_runs/lo-normative-accuracy-5/` for local analysis, or use:

```powershell
python scripts/export_lo_kaggle_results.py --download
```

In [ ]:
if SKIP_DRY_RUN_FOR_BUILD:
    print("Skipping download preview (SKIP_DRY_RUN_FOR_BUILD=True)")
else:
    import pandas as pd

    for path in (merged_path, pivot_path):
        print(path, f"({path.stat().st_size:,} bytes)")

    merged_df = pd.read_csv(merged_path)
    expected_n = DEBUG_MAX_PROMPTS or EXPECTED_BENCHMARK_PROMPTS
    empty_responses = int(
        merged_df["response"].astype(str).str.strip().eq("").sum()
    ) if "response" in merged_df.columns else 0
    blank_scores = int(
        merged_df["score"].isna().sum()
        + (
            merged_df["score"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["", "nan", "none"])
            & merged_df["score"].notna()
        ).sum()
    )
    pivot_df = pd.read_csv(pivot_path, index_col=0) if pivot_path.is_file() else None
    pivot_nan = int(pivot_df.isna().sum().sum()) if pivot_df is not None else 0

    print(f"\nMerged rows: {len(merged_df)}  |  model: {selected_llm_name}")
    print(
        "Naive LP confusion rows:",
        int(merged_df["naive_lp_confusion"].astype(str).str.lower().eq("true").sum()),
    )
    print(
        f"Coverage: {len(merged_df) - empty_responses} answered / "
        f"{len(merged_df)} merged rows / {expected_n} expected prompts"
    )
    print(f"Empty responses (missing scores): {empty_responses}")
    print(f"Blank/NaN score fields: {blank_scores}")
    print(
        f"NaN cells in score pivot: {pivot_nan} / "
        f"{pivot_df.size if pivot_df is not None else 0}"
    )

    try:
        from IPython.display import FileLink, display

        display(
            FileLink(merged_path.name, result_html_prefix="Download merged: "),
            FileLink(pivot_path.name, result_html_prefix="Download pivot: "),
        )
    except Exception:
        print("Open the Files panel and download from /kaggle/working/")

    merged_df.head()

In [ ]:
import kaggle_benchmarks as kbench
import kaggle_benchmarks.ui.ipython_magics  # registers %choose in interactive sessions

from benchmarks.lp_rate_tasks import lo_normative_accuracy_5

# Dry-run alone does not create the task .run.json; this call does.
run = lo_normative_accuracy_5.run(kbench.llm)
print("Task score:", run.result)
print("Task passed:", run.passed)

%choose lo_normative_accuracy_5